# convT-as-flipped-padded-conv — faded example 1: Complete the kernel transform in the ConvT rebuild

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-as-flipped-padded-conv`. The last cell reports your progress on the `CNN: ConvT as flipped padded conv` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT as flipped padded conv` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convT-as-flipped-padded-conv`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convT-as-flipped-padded-conv"
DD_SUBTOPIC = "CNN: ConvT as flipped padded conv"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Stride-1 `F.conv_transpose2d(x, weight)` equals `F.conv2d` on an input padded by `K-1`, using the kernel flipped along both spatial axes and with its `(IC, OC)` channel axes swapped to `(OC, IC)`. The padding is the easy part; the flip-and-swap of the kernel is the conceptually load-bearing step.

## Faded exercise 1

Implement `convT_as_padded_conv(x, weight)` so it reproduces `F.conv_transpose2d(x, weight)` (stride 1, no padding) using only `F.conv2d`. The input padding by `K-1` is already written for you. You must complete the **kernel transform**: starting from the ConvT-layout `weight` of shape `(IC, OC, K, K)`, produce the conv2d-ready kernel of shape `(OC, IC, K, K)` that makes the equivalence hold.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch.nn.functional as F

def convT_as_padded_conv(x, weight):
    K = weight.shape[-1]
    x_pad = F.pad(x, (K - 1, K - 1, K - 1, K - 1))
    w = weight.flip(-1).flip(-2).transpose(0, 1)
    return F.conv2d(x_pad, w)

t.manual_seed(0)
x = t.randn(2, 3, 5, 5)
weight = t.randn(3, 4, 3, 3)
out = convT_as_padded_conv(x, weight)
print(tuple(out.shape))


def _test():
    import torch.nn.functional as F
    t.manual_seed(0)
    x = t.randn(2, 3, 5, 5)
    weight = t.randn(3, 4, 3, 3)
    got = convT_as_padded_conv(x, weight)
    ref = F.conv_transpose2d(x, weight)
    assert got.shape == ref.shape, (got.shape, ref.shape)
    assert tuple(got.shape) == (2, 4, 7, 7), got.shape
    assert t.allclose(got, ref, atol=1e-5), (got - ref).abs().max().item()


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn.functional as F

def convT_as_padded_conv(x, weight):
    K = weight.shape[-1]
    x_pad = F.pad(x, (K - 1, K - 1, K - 1, K - 1))
    w = weight.flip(-1).flip(-2).transpose(0, 1)
    return F.conv2d(x_pad, w)

t.manual_seed(0)
x = t.randn(2, 3, 5, 5)
weight = t.randn(3, 4, 3, 3)
out = convT_as_padded_conv(x, weight)
print(tuple(out.shape))
```
</details>